In [ ]:
import requests
import urllib.error, urllib.parse
from urllib.parse import urljoin
from bs4 import BeautifulSoup
import re
from google.colab import drive
from PIL import Image
from io import BytesIO
import os

drive_root_dir ='/content/drive'
drive.mount(drive_root_dir)
save_folder = '/My Drive/Private/xxx/downloads/'
target_folder = drive_root_dir + save_folder

if not os.path.exists(target_folder):
  os.mkdir(target_folder)

targets = [
  {
    'folder_name': 'kikuchi-hina',    # 保存先のフォルダ名
    'file_name': 'kikuchi-hina',      # 保存時のファイル名：空の場合はオリジナルのファイルの名で保存、ファイル名を指定した場合は、{file_name}000 + 拡張子の形式で保存
    'url': 'http://intervaluesk.com/k/kikutihimena1.html' # ダウンロード対象のサイトURL
  },
]
pattern1 = ".+(jpg|png|bmp)$"
pattern2 = "^http"

try:
  for item in targets:
    url = item['url']
    base_file_name = item['file_name']
    save_folder = target_folder + item['folder_name'] + '/'

    # フォルダの存在をチェックし、無ければ作成する
    if not os.path.exists(save_folder):
      os.mkdir(save_folder)

    html = urllib.request.urlopen(url)
    soup = BeautifulSoup(html, "html.parser")
    # links = [link.get('href') for link in soup.find_all('a')]
    links = [link.get('src') for link in soup.find_all('img')]
    counter = 1
    for link in links:
      if not link is None:
        if re.search(pattern2, link, re.IGNORECASE) is None:
          link = urljoin(url,link)
        if not re.search(pattern1, link, re.IGNORECASE) is None:
          print(link)
          # list = link.split("/")
          filename, ext =  os.path.splitext(link)
          if base_file_name != '':
            filename = base_file_name + str(counter).zfill(3)
          filename = filename + ext
          # print(filename)
          response = requests.get(link)
          img = Image.open(BytesIO(response.content))
          img.save(save_folder + filename)
  print('complete')

except Exception as e:
  print(e)